# 00 · Datos: leer y revisar ATUS (INEGI)

**Qué hace:** lee el Excel de accidentes, revisa que venga bien, corrige los acentos y guarda solo los accidentes de la Zona Metropolitana de Guadalajara (ZMG) de 2022 a 2024.

**Entra:** `ATUS_JAL_2019_2024.xlsx` (en esta carpeta o en `data/`)
**Sale:** `data/accidentes_zmg.csv`, un renglón por accidente.

In [1]:
import pandas as pd
from pathlib import Path

ANIO_MIN = 2022                      # recomendación del tutor: 2022–2023 para entrenar y 2024 para probar
MUNICIPIOS_ZMG = {39: "Guadalajara", 120: "Zapopan", 98: "Tlaquepaque", 101: "Tonalá", 97: "Tlajomulco",
                  70: "El Salto", 44: "Ixtlahuacán de los Membrillos", 51: "Juanacatlán", 124: "Zapotlanejo",
                  2: "Acatlán de Juárez"}
Path("data").mkdir(exist_ok=True)

## 1. Leer el Excel

In [2]:
ruta = next(p for p in list(Path(".").glob("ATUS_JAL*.xls*")) + list(Path("data").glob("ATUS_JAL*.xls*")))
df = pd.read_excel(ruta)
df.columns = df.columns.str.strip().str.upper()
print(f"{ruta}: {len(df):,} renglones · {df.shape[1]} columnas")
df.head()

ATUS_JAL_2019_2024.xlsx: 27,928 renglones · 50 columnas


,ID,EDO,MES,ANIO,MPIO,HORA,MINUTOS,DIA,DIASEMANA,URBANA,...,OTROHERIDO,TOTMUERTOS,TOTHERIDOS,CLASE,CALLE1,CALLE2,CARRETERA,LONGITUD,LATITUD,OID
0,1182475-999-3,14,1,2024,23,3,20,1,1,2,...,0,0,0,3,FEDERICO DEL TORO (EJE),FEDERICO DEL TORO (EJE),NaN,-103.462184,19.716718,88718
1,1182475-999-4,14,1,2024,23,6,10,1,1,1,...,0,0,0,3,CONSTITUCI”N,CALLE DEL CARRO,NaN,-103.467932,19.699383,88719
2,1182475-999-5,14,1,2024,23,7,15,1,1,1,...,0,0,2,2,MOCTEZUMA,IGNACIO L”PEZ RAY”N,NaN,-103.460387,19.707949,88720
3,1182475-999-6,14,1,2024,23,19,17,2,2,1,...,0,0,0,3,MIGUEL HIDALGO,FRAY BARTOLOM… DE LAS CASAS,NaN,-103.463201,19.721269,88721
4,1182475-999-7,14,1,2024,23,19,28,2,2,1,...,0,0,0,3,OBISPO SERAFÕN V¡ZQUEZ ELIZALDE,DE GORDIANO GUZM¡N,NaN,-103.470893,19.679771,88722


## 2. Revisiones
- ¿Están las columnas que vamos a usar?
- ¿Hay fechas inválidas o accidentes sin coordenadas?
- ¿Cuántos accidentes hay por municipio y año?

In [3]:
COLUMNAS = ["ID", "MPIO", "ANIO", "MES", "DIA", "HORA", "CLASE", "TIPACCID", "CALLE1", "CALLE2",
            "LATITUD", "LONGITUD", "TOTMUERTOS", "TOTHERIDOS"]
print("Columnas que faltan:", [c for c in COLUMNAS if c not in df.columns] or "ninguna")

df["FECHA"] = pd.to_datetime(dict(year=df.ANIO, month=df.MES, day=df.DIA), errors="coerce")
print("Fechas inválidas:", df.FECHA.isna().sum())
print("Sin coordenadas:", df[["LATITUD", "LONGITUD"]].isna().any(axis=1).sum())

municipio = df.MPIO.map(MUNICIPIOS_ZMG).fillna("Fuera de la ZMG")
pd.crosstab(municipio, df.ANIO, margins=True, margins_name="Total")

Columnas que faltan: ninguna
Fechas inválidas: 0
Sin coordenadas: 0


ANIO,2022,2023,2024,Total
MPIO,,,,
El Salto,18,52,41,111
Fuera de la ZMG,985,858,1213,3056
Guadalajara,3735,3543,4290,11568
Ixtlahuacán de los Membrillos,1,6,4,11
Tlajomulco,271,393,486,1150
Tlaquepaque,970,1030,1067,3067
Tonalá,257,300,365,922
Zapopan,2315,2596,3132,8043
Total,8552,8778,10598,27928


## 3. Corregir acentos
Los nombres de calles vienen mal codificados: `L”PEZ` en vez de `LÓPEZ`, `—` en vez de `Ñ`. Se reemplaza cada carácter raro por el correcto.

In [4]:
ARREGLO = str.maketrans({"¡": "Á", "…": "É", "Õ": "Í", "”": "Ó", "⁄": "Ú", "—": "Ñ", "‹": "Ü",
                         "·": "á", "È": "é", "Ì": "í", "Û": "ó", "˙": "ú", "Ò": "ñ"})
print("Antes:", df[df.CALLE1.astype(str).str.contains("MATEOS")].CALLE1.iloc[0])
for c in ["CALLE1", "CALLE2"]:
    df[c] = df[c].map(lambda v: v.translate(ARREGLO) if isinstance(v, str) else v)
print("Después:", df[df.CALLE1.astype(str).str.contains("MATEOS")].CALLE1.iloc[0])

Antes: L”PEZ MATEOS
Después: LÓPEZ MATEOS


## 4. Filtrar la ZMG desde 2022 y guardar

In [5]:
zmg = df[df.MPIO.isin(MUNICIPIOS_ZMG) & (df.ANIO >= ANIO_MIN) & df.FECHA.notna()].copy()
zmg["MUNICIPIO"] = zmg.MPIO.map(MUNICIPIOS_ZMG)
zmg.to_csv("data/accidentes_zmg.csv", index=False)
print(f"→ data/accidentes_zmg.csv: {len(zmg):,} accidentes de la ZMG, {ANIO_MIN}–{zmg.ANIO.max()}")
zmg.groupby("ANIO").size()

→ data/accidentes_zmg.csv: 24,872 accidentes de la ZMG, 2022–2024


ANIO
2022    7567
2023    7920
2024    9385
dtype: int64

**Qué vemos:**
- El archivo dice 2019–2024, pero trae 2022–2024.
- 2024 sube bastante. Parte es un cambio en cómo se registran los accidentes, no necesariamente más choques.